# Topic Modeling
### Bert Sentence Embedding, LDA, and K-means Clustering

#### Concatenate the sentence embeddings with LDA, use an autoencoder to get the latent representation and perform K-means to perform clustering on the data

#### After the sentences have been assigned to their clusters, we can extract the candidate topics for each cluster using tokens of it's sentences

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')


In [ ]:
% cd /content/drive/My Drive/Thesis_colab

In [ ]:
! pip install transformers
! pip install pandas
! pip install torch
! pip install nltk
! pip install gensim
! pip install matplotlib
! pip install umap-learn
! pip install wordcloud
! pip install sklearn
! pip install tqdm
! pip install stop_words
! pip install xlrd
! pip install langdetect
! pip install openpyxl
! pip install sentence_transformers
! pip install pyLDAvis
! pip install pickle5


In [ ]:
# install ocra to save figures using plotly
!pip install plotly==4.7.1
!wget https://github.com/plotly/orca/releases/download/v1.2.1/orca-1.2.1-x86_64.AppImage -O /usr/local/bin/orca
!chmod +x /usr/local/bin/orca
!apt-get update
!apt-get install xvfb libgtk2.0-0 libgconf-2-4


In [1]:
import nltk
import numpy as np
import statistics
from sklearn.decomposition import PCA
from collections import Counter
import pandas as pd

In [ ]:
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

### 1. Reading and preprocessing the data

In [ ]:
from utilities.document_preprocessing import preprocess_documents

#### The preprocess_document function reads the sentences from the file and generates tokens from each sentences.

In [2]:
Verbose = False

In [ ]:
# Read new data file
file = "data/PMP_Skills_Matching_Pilot_2018_2019_small.xlsx"

#ids, sentences, token_lists = preprocess_documents(file, verbose = Verbose)


#ids = list (ids)
#num_docs = len(ids)

In [3]:
#read the previously processed data

#data_file = "output/preprocessed_data/use_this_data.pkl"
#data_file = "output/11.09.2020/mercks_objectives/use_this_data_merck_unique.pkl"
data_file = "output/preprocessed_data/job/use_this_data_10000.pkl"

import pickle5 as pickle

with (open(data_file, "rb")) as openfile:
    data = pickle.load(openfile)

sentences = list(data['sentences'])
token_lists = list(data['token_list'])
ids = list(data['id'])
num_docs = len(ids)


In [4]:
max = 0
for sent in sentences:
    if len(sent.split()) > max:
        max = len(sent.split())
print (max)

321


## 2. Topic Modeling

In [5]:
from models.topic_model import Topic_Model

#### The topic_Model class does the follow
- Extract Bert sentence embeddings for all sentences (Local features)
- get topic distribution for each sentence using LDA/LSI
- concatenate both and learn the latent representation using an auto-encoder
- cluster the latent representation into K clusters

In [6]:
# define the numbe of clusters
k = 10

In [9]:
# Define the topic model object
TopicModelObject = Topic_Model(base = 'LDA', k = k, verbose = Verbose, embedding = 'bert',
                               saved_embedding_dir = None, 
                               embedding_batch_size = 500)

In [ ]:
# for Colab
# Define the topic model object
TopicModelObject = Topic_Model( k = k, verbose = Verbose, embedding = 'bert', lamda = 15,
                               saved_embedding_dir = None, embedding_batch_size = 100)

In [10]:
# Fit the topic model
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    warnings.warn("deprecated", DeprecationWarning)


    TopicModelObject.fit(sentences, token_lists)

Getting vector representations for LDA 
Getting vector representations for LDA. Done! 


 Getting vector representations for bert 
cpu
Embedding for batch 0 out of 19 batches Completed.


KeyboardInterrupt: 

In [ ]:
TopicModelObject.vec

## 3. Analysis on learned features

#### Preprocessing (Dimensionality Reduction) for visualization

Perform Dimensionality reduction on the learned features using the following steps:

1) Merge all the vectors of each employee into one by taking average

2) Assign the class label to each employee based on argmax on label counts.

3) Further compress the average feature vector into 2d/3d space for visualization.

In [ ]:
# Getting feature vectors and cluster labels assigned by the model
if TopicModelObject.method == 'LDA' or TopicModelObject.method =='LSI':
    feature_vector = TopicModelObject.vec[TopicModelObject.method]
    # for each sentence, find out the cluster it belongs to
    cluster_label_per_document = np.argmax(np.array(feature_vector), axis=1)
else:
    cluster_label_per_document = np.array(TopicModelObject.cluster_model.labels_)
    feature_vector = TopicModelObject.vec[TopicModelObject.method]
    feature_vector = feature_vector.cpu().numpy()
    

unique_ids = list(set(ids))
num_employee = len(unique_ids)

In [ ]:
# Create place holder for the analysis
averaged_employee_features = pd.DataFrame(None, index=pd.Index(unique_ids, name='Employee id'),
                                          columns=pd.Index(
                                              ['feature_vector', 'cluster', 'feature_vector_2d', 'feature_vector_3d'],
                                              name='Compressed_features'))

In [ ]:
# get the indices of all objectives/documents/sentence for each employee (for later use)
employee_objective_indices = dict()
for employee in unique_ids:
    employee_indices = [i for i, e in enumerate(ids) if (ids[i] == employee) ]
    employee_objective_indices[employee] = employee_indices

In [ ]:
# 1 & 2 take average of all feature vectors belonging to an employee and assigning cluster label by taking the mode

for employee in unique_ids:
    #get all feature vectors belonging to an employee
    feature_vector_per_employee = feature_vector[employee_objective_indices[employee]]
    # get label for all feature vectors belonging to that employee
    clusters_per_employee = cluster_label_per_document[employee_objective_indices[employee]]  
    
    #take mean of all feature vectors
    averaged_employee_features['feature_vector'][employee] = np.mean(feature_vector_per_employee, axis=0)
    
    # pick the label with most occurance for that employee
    c = Counter(clusters_per_employee)
    averaged_employee_features['cluster'][employee] = c.most_common(1)[0][0]
    
averaged_employee_features

In [ ]:
# 3) doing dimensionality reduction into 2d/3d Space
from sklearn.manifold import TSNE

#todo: find a more permanent solution for this
# panda returns a array of array for the column feature vector. Converting from array of array to 2d array
feature_vector_ = np.stack(averaged_employee_features['feature_vector'].to_numpy())

# reduce dimension of feature vectors into 2
feature_vector_2d = TSNE(n_components=2).fit_transform(feature_vector_)
averaged_employee_features['feature_vector_2d'] = list(feature_vector_2d)

# reduce dimension of feature vectors into 3
feature_vector_3d = TSNE(n_components=3).fit_transform(feature_vector_)
averaged_employee_features['feature_vector_3d'] = list(feature_vector_3d)


averaged_employee_features

## 4.1 Visualization: Word Cloud

In [ ]:
from utilities.visualizations import get_wordcloud
from utilities.utils import generate_n_gram
%matplotlib inline

In [ ]:
#Getting indexes of sentences belonging to each cluster
sentence_label_dict = dict()

if TopicModelObject.method == 'LDA':
    labels = np.argmax(TopicModelObject.vec['LDA'], axis=1)
else:
    labels = TopicModelObject.cluster_model.labels_

for label in set(labels):
    sentence_label_dict[label] = [idx for idx, l in enumerate(labels) if l == label]

In [ ]:
TopicModelObject.dir 

In [ ]:
### Top words from LDA
if TopicModelObject.method == 'LDA':
    
    lda_class_distribution = TopicModelObject.vec['LDA']
    labels = np.argmax(lda_class_distribution, axis=1)
    

    # To get the top words for each topic, this method only uses LDA, doesn't utilize Bert or K mean clustering
    TopicModelObject.get_top_words_lda(top=10)

    for k , cluster_word_distribution in TopicModelObject.get_top_words_lda(top=100).items():
        get_wordcloud(dir = TopicModelObject.dir, word_distribution = cluster_word_distribution, topic = k )
# Get top words from autoencoder (LDA + Embeddings) and create word clouds on n grams
else:
    grams = 1
    for ng in range(grams):
        print ("WordClouds for {} grams".format(str(ng+1)))
        for i in range(TopicModelObject.k):

            k_sentences = np.array(sentences)[sentence_label_dict[i]]
            ngram_token_with_frequency = generate_n_gram(k_sentences, n=ng+1, separator = '-' ,
                                        keep_stopwords= False, stem=False, keep_nouns_only=True)
    
            get_wordcloud(topic=i, word_count_dict = ngram_token_with_frequency, dir = TopicModelObject.dir)
    

In [ ]:
TopicModelObject.dir

## 4.2 Visualization: Scatter plot

Visualize the feature vectors of each employee in a 2d/3d space using plotly

In [ ]:
import plotly.graph_objects as go
from sklearn import preprocessing

In [ ]:
from collections import Counter
#main topics for each cluster
cluster_titles = []

for i in range(TopicModelObject.k):
    k_sentences = np.array(sentences)[sentence_label_dict[i]]
    ngram_token_with_frequency = Counter(generate_n_gram(k_sentences, n=2, separator = '-' ,
                                        keep_stopwords= False, stem=False, keep_nouns_only=False))
    
    
    cluster_title = ', '.join([key for key, value in ngram_token_with_frequency.most_common(6)])
    
    cluster_titles.append(cluster_title)
cluster_titles

In [ ]:
np_sentences = np.array(sentences)

In [ ]:
fig = go.Figure()


for i, cl in enumerate(range(k)):
    
    cluster_df = averaged_employee_features[averaged_employee_features.cluster==cl]
    #plot all the employees belonging to a cluster using a new trace. 
    features = np.stack(cluster_df['feature_vector_2d'].to_numpy())

    employee_titles = []
    # constructing hover text (all objectives belonging to that employee) 
    for emp in cluster_df.index.values:
        
        emp_indices = employee_objective_indices[emp]
        employee_title = '. '.join([objective for objective in np_sentences[emp_indices]])
    
        employee_titles.append(emp + ': ' + employee_title)
        
    fig.add_trace(go.Scattergl(
      x=features[:, 0],
      y=features[:, 1],
      mode='markers',
      showlegend=True,
      text = employee_titles,
      marker=dict(size=8, color=i),
      name="cluster_"+str(cl+1) +": " + cluster_titles[cl]
      ))



fig.update_layout(title='Distribution of employees over clusters in 2d space', legend=dict(
        x=0.6,
        y=1.4,
        traceorder='normal',
        font=dict(
            size=10)))
fig.show()

path = TopicModelObject.dir + '2D_clusters.jpeg'
fig.write_image(path)


In [ ]:
fig = go.Figure()


for i, cl in enumerate(range(k)):
    cluster_df = averaged_employee_features[averaged_employee_features.cluster==cl]

    #plot all the employees belonging to a cluster using a new trace. 
    features = np.stack(cluster_df['feature_vector_2d'].to_numpy())
    
    #normalize the features between 0 and 1
    features_normed = preprocessing.normalize(features)
    # Normalize the features between 0 and 1 before visualization
    min_ = features.min(axis=0)
    max_ = features.max(axis=0)
    features_normed = (features - min_) / (max_ - min_)
    
    employee_titles = []
    # constructing hover text (all objectives belonging to that employee) 
    for emp in cluster_df.index.values:
        
        emp_indices = employee_objective_indices[emp]
        employee_title = '. '.join([objective for objective in np_sentences[emp_indices]])
    
        employee_titles.append(emp + ': ' + employee_title)


    fig.add_trace(go.Scattergl(
      x=features_normed[:, 0],
      y=features_normed[:, 1],
      mode='markers',
      showlegend=True,
      text = employee_titles,
      marker=dict(size=8, color=i),
      name="cluster_"+str(cl+1) +": " + cluster_titles[cl]
    ))



fig.update_layout(title='Distribution of employees over clusters in 2d space - Normalized', legend=dict(
        x=0.6,
        y=1.4,
        traceorder='normal',
        font=dict(
            size=10)))
fig.show()
path = TopicModelObject.dir + '2D_clusters_norm.jpeg'
fig.write_image(path)

In [ ]:
fig = go.Figure()
for i , cl in enumerate(range(k)):
    
    cluster_df = averaged_employee_features[averaged_employee_features.cluster==cl]

    #plot all the employees belonging to a cluster using a new trace. 
    features = np.stack(cluster_df['feature_vector_3d'].to_numpy())
    
    employee_titles = []
    # constructing hover text (all objectives belonging to that employee) 
    for emp in cluster_df.index.values:
        
        emp_indices = employee_objective_indices[emp]
        employee_title = '. '.join([objective for objective in np_sentences[emp_indices]])
    
        employee_titles.append(emp + ': ' + employee_title)


    fig.add_trace(go.Scatter3d(
      x=features[:, 0],
      y=features[:, 1],
      z=features[:, 2],
      mode='markers',
      showlegend=True,
      text = employee_titles,
      marker=dict(size=3, color=i),
      name="cluster_"+str(cl+1) +": " + cluster_titles[cl]
    ))

fig.update_layout(title='Distribution of employees over clusters in 3d Space', legend=dict(
        x=0.6,
        y=1.4,
        traceorder='normal',
        font=dict(
            size=10)))
fig.show()
path = TopicModelObject.dir + '3D_clusters.jpeg'
fig.write_image(path)

In [ ]:
fig = go.Figure()

for i, cl in enumerate(range(k)):
    cluster_df = averaged_employee_features[averaged_employee_features.cluster==cl]

    #plot all the employees belonging to a cluster using a new trace. 
    features = np.stack(cluster_df['feature_vector_3d'].to_numpy())
    
    #normalize the features between 0 and 1
    
    
    min_ = features.min(axis=0)
    max_ = features.max(axis=0)

    features_normed = (features - min_) / (max_ - min_)
    #features_normed = preprocessing.normalize(features)

    
    employee_titles = []
    # constructing hover text (all objectives belonging to that employee) 
    for emp in cluster_df.index.values:
        
        emp_indices = employee_objective_indices[emp]
        employee_title = '. '.join([objective for objective in np_sentences[emp_indices]])
    
        employee_titles.append(emp + ': ' + employee_title)


    fig.add_trace(go.Scatter3d(
      x=features_normed[:, 0],
      y=features_normed[:, 1],
      z=features_normed[:, 2],
      mode='markers',
      showlegend=True,
      text = employee_titles,
      marker=dict(size=3, color=i),
      name="cluster_"+str(cl+1) +": " + cluster_titles[cl]
    ))

fig.update_layout(title='Distribution of employees over clusters in 3d Space - Normalized', legend=dict(
        x=0.6,
        y=1.4,
        traceorder='normal',
        font=dict(
            size=10)))
fig.show()
path = TopicModelObject.dir + '3d_clusters_norm1.jpeg'
fig.write_image(path)

### 4.3 pyLDAvis
This visualization only works on LDA results

In [ ]:
import pyLDAvis
from pyLDAvis import gensim as gs

In [ ]:
id2word = corpora.Dictionary(token_lists)
corpus = [id2word.doc2bow(text) for text in token_lists]

In [ ]:
pyLDAvis.enable_notebook()
vis = gs.prepare(TopicModelObject.ldamodel, corpus, id2word)
vis